# 🏠 Housing Price Prediction — LightGBM
This notebook trains a **LightGBM** gradient boosting model on the California Housing dataset.  
Uses native categorical support + early stopping — same approach as the AQI LightGBM model.  
Upload your `housing.csv` from your local machine using the file picker below.

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q lightgbm scikit-learn pandas numpy matplotlib seaborn joblib

In [ ]:
# ── Upload dataset from local machine ────────────────────────────────────────
from google.colab import files

print('📂 Please select your housing.csv file...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f'✅ Uploaded: {filename}')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import io
import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded ✅')

In [ ]:
# ── Load & preview data ───────────────────────────────────────────────────────
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print('Shape:', df.shape)
df.head()

In [ ]:
# ── Basic info & missing values ───────────────────────────────────────────────
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
# ── Preprocessing ─────────────────────────────────────────────────────────────
df = df.dropna()

feature_cols = [
    'longitude', 'latitude', 'housing_median_age',
    'total_rooms', 'total_bedrooms', 'population',
    'households', 'median_income', 'ocean_proximity'
]
cat_cols = ['ocean_proximity']

X = df[feature_cols]
y = df['median_house_value']

print('Features:', feature_cols)
print('Target: median_house_value')

In [ ]:
# ── Train / Test split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Cast categorical columns for LightGBM native handling
for c in cat_cols:
    X_train[c] = X_train[c].astype('category')
    X_test[c]  = X_test[c].astype('category')

print(f'Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}')

In [ ]:
# ── Train LightGBM (same hyperparams as AQI model) ───────────────────────────
model = lgb.LGBMRegressor(
    objective='regression',
    metric='rmse',
    boosting_type='gbdt',
    num_leaves=31,
    learning_rate=0.05,
    n_estimators=1000,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    random_state=42,
    verbose=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric='rmse',
    categorical_feature=cat_cols,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
)

y_pred = model.predict(X_test, num_iteration=model.best_iteration_)

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print(f'\nMSE  : {mse:,.2f}')
print(f'RMSE : {rmse:,.2f}')
print(f'R²   : {r2:.4f}')
print(f'Best iteration: {model.best_iteration_}')

## 📊 Visualizations

In [ ]:
# ── 1. Actual vs Predicted ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred, alpha=0.3, edgecolors='k', linewidths=0.4, color='seagreen')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual House Value ($)', fontsize=12)
ax.set_ylabel('Predicted House Value ($)', fontsize=12)
ax.set_title('LightGBM — Actual vs Predicted', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 2. Residuals Distribution ────────────────────────────────────────────────
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(residuals, bins=60, color='seagreen', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Residuals Distribution', fontsize=13)
axes[0].set_xlabel('Residual')
axes[0].set_ylabel('Count')

axes[1].scatter(y_pred, residuals, alpha=0.3, color='seagreen', edgecolors='k', linewidths=0.3)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Residuals vs Fitted Values', fontsize=13)
axes[1].set_xlabel('Fitted Values')
axes[1].set_ylabel('Residuals')

plt.tight_layout()
plt.show()

In [ ]:
# ── 3. Feature Importance ────────────────────────────────────────────────────
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model.feature_importances_
}).sort_values('Importance')

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(importance_df['Feature'], importance_df['Importance'],
        color='seagreen', edgecolor='white')
ax.set_title('LightGBM — Feature Importance (split count)', fontsize=14)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4. Training Loss Curve (RMSE over boosting rounds) ───────────────────────
results = model.evals_result_
train_rmse = results.get('training', {}).get('rmse', [])
valid_rmse = results.get('valid_0', {}).get('rmse', [])

if valid_rmse:
    fig, ax = plt.subplots(figsize=(9, 5))
    rounds = range(1, len(valid_rmse) + 1)
    if train_rmse:
        ax.plot(rounds, train_rmse, label='Train RMSE', color='steelblue')
    ax.plot(rounds, valid_rmse, label='Validation RMSE', color='seagreen')
    ax.axvline(model.best_iteration_, color='red', linestyle='--',
               label=f'Best iteration ({model.best_iteration_})')
    ax.set_title('LightGBM — RMSE over Boosting Rounds', fontsize=13)
    ax.set_xlabel('Boosting Round')
    ax.set_ylabel('RMSE')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No eval results found — rerun with eval_set provided.')

In [ ]:
# ── 5. Geographic scatter — Predicted house value on map ─────────────────────
# Build a test dataframe with coordinates and predictions
geo_df = X_test[['longitude', 'latitude']].copy()
geo_df['predicted'] = y_pred
geo_df['actual']    = y_test.values

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, col, title, cmap in zip(
    axes,
    ['actual', 'predicted'],
    ['Actual House Value', 'Predicted House Value (LightGBM)'],
    ['YlOrRd', 'YlGn']
):
    sc = ax.scatter(
        geo_df['longitude'], geo_df['latitude'],
        c=geo_df[col], cmap=cmap, alpha=0.4, s=5
    )
    plt.colorbar(sc, ax=ax, label='Value ($)')
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

plt.suptitle('California Housing — Geographic Value Distribution', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Save model ────────────────────────────────────────────────────────────────
joblib.dump(model, 'housing_lgbm_model.pkl')
print('Model saved as housing_lgbm_model.pkl')
files.download('housing_lgbm_model.pkl')